# Pré-processamento SPOS

Este notebook contém um pipeline para pré-processar a base SPOS.

Funcionalidades:
- lê imagens `.bmp` recursivamente em `DATA_DIR` (ignora arquivos `._*` do macOS)
- suporte para modalidades `nir` e `vis` (processa ambas por padrão)
- leitura + resize paralelos (ThreadPool)
- agrupa frames por pasta de sequência e usa anotações `onset`→`apex` se fornecidas
- fallback heurístico (janela central) quando não há anotações
- pad/truncate das sequências para comprimento fixo por tipo (posed/spontaneous)
- codifica labels, salva `X.npy`, `y.npy`, `label_mapping.json` e `X_padded.npy`

Antes de rodar, ajuste `DATA_DIR` e `OUTPUT_ROOT` nas próximas células.

In [13]:
# Imports e configuração de caminhos/parâmetros (modelo SMIC)
from pathlib import Path
import os, time, json
import numpy as np
import cv2
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict, Counter
from sklearn.preprocessing import LabelEncoder

# Ajuste estes caminhos antes de executar
DATA_DIR = Path('/Volumes/Dados/tcc/SPOS database')
OUTPUT_DIR = Path('../../data/SPOS/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Parâmetros
IMG_SIZE = (112, 112)
MAX_FRAMES = 16  # padrão para padding/truncate (ajuste conforme necessário)
num_workers = min(8, (os.cpu_count() or 1) * 2)

print('CONFIG: DATA_DIR=', DATA_DIR, 'OUTPUT_DIR=', OUTPUT_DIR, 'IMG_SIZE=', IMG_SIZE, 'num_workers=', num_workers)


CONFIG: DATA_DIR= /Volumes/Dados/tcc/SPOS database OUTPUT_DIR= ../../data/SPOS/processed IMG_SIZE= (112, 112) num_workers= 8


In [14]:
# Benchmark rápido (amostra) — mede tempo médio de leitura+resize e conta .bmp válidos
import time
SAMPLE_N = 200

# encontrar até SAMPLE_N arquivos válidos (ignora arquivos que começam com ._)
paths = []
for p in DATA_DIR.glob('**/*.bmp'):
    if not p.name.startswith('._') and p.is_file():
        paths.append(p)
    if len(paths) >= SAMPLE_N:
        break

print('Amostra coletada:', len(paths))
if not paths:
    print('Nenhuma imagem .bmp válida encontrada para amostra.')
else:
    t0 = time.perf_counter()
    valid = 0
    for p in paths:
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, IMG_SIZE)
        valid += 1
    t1 = time.perf_counter()
    if valid == 0:
        print('Nenhuma imagem válida lida na amostra.')
    else:
        per_img = (t1 - t0) / valid
        total_bmps = sum(1 for f in DATA_DIR.glob('**/*.bmp') if not f.name.startswith('._') and f.is_file())
        est_total_s = per_img * total_bmps
        print(f'Tempo médio por imagem (leitura+resize): {per_img:.4f} s')
        print(f'Total .bmp válidos encontrados: {total_bmps}')
        print(f'Estimativa total: {est_total_s/60:.1f} minutos ({est_total_s:.0f} segundos)')


Amostra coletada: 200
Tempo médio por imagem (leitura+resize): 0.0560 s
Total .bmp válidos encontrados: 22462
Estimativa total: 21.0 minutos (1258 segundos)
Tempo médio por imagem (leitura+resize): 0.0560 s
Total .bmp válidos encontrados: 22462
Estimativa total: 21.0 minutos (1258 segundos)


In [15]:
# Processamento paralelo (ThreadPool) — lê e redimensiona imagens e reconstrói sequências (modelo SMIC)
from collections import defaultdict

# coletar todos os caminhos válidos (ignora arquivos '._')
all_paths = [f for f in DATA_DIR.glob('**/*.bmp') if not f.name.startswith('._') and f.is_file()]
print(f'Total imagens válidas encontradas: {len(all_paths)}')

if not all_paths:
    print('Nenhuma imagem .bmp válida encontrada — verifique o caminho')
else:
    def read_resize(path):
        try:
            img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                return (path, None)
            img = cv2.resize(img, IMG_SIZE)
            return (path, img)
        except Exception:
            return (path, None)

    t0 = time.perf_counter()
    results = {}  # path -> img array
    with ThreadPoolExecutor(max_workers=num_workers) as ex:
        futures = {ex.submit(read_resize, p): p for p in all_paths}
        for i, fut in enumerate(as_completed(futures), 1):
            p, img = fut.result()
            if img is not None:
                results[p] = img
            if i % 1000 == 0:
                print(f'Processados {i}/{len(all_paths)}')
    t1 = time.perf_counter()
    successful = len(results)
    print(f'Leitura+resize paralelos concluídos: {successful}/{len(all_paths)} imagens válidas lidas em {t1-t0:.1f}s')

    # reagrupar por pasta de vídeo e reconstruir sequências ordenadas
    grouped = defaultdict(list)
    for p, img in results.items():
        grouped[p.parent].append((p.name, img))
    frames_all = []
    labels_all = []
    for subj_path, items in grouped.items():
        items.sort(key=lambda x: x[0])
        seq = [img for _, img in items]
        if not seq:
            continue
        seq = np.stack(seq)
        frames_all.append(seq)
        labels_all.append(subj_path.parent.name)
    print(f'Total de sequências reconstruídas: {len(frames_all)}')


Total imagens válidas encontradas: 22462
Processados 1000/22462
Processados 1000/22462
Processados 2000/22462
Processados 2000/22462
Processados 3000/22462
Processados 3000/22462
Processados 4000/22462
Processados 4000/22462
Processados 5000/22462
Processados 5000/22462
Processados 6000/22462
Processados 6000/22462
Processados 7000/22462
Processados 7000/22462
Processados 8000/22462
Processados 8000/22462
Processados 9000/22462
Processados 9000/22462
Processados 10000/22462
Processados 10000/22462
Processados 11000/22462
Processados 11000/22462
Processados 12000/22462
Processados 12000/22462
Processados 13000/22462
Processados 13000/22462
Processados 14000/22462
Processados 14000/22462
Processados 15000/22462
Processados 15000/22462
Processados 16000/22462
Processados 16000/22462
Processados 17000/22462
Processados 17000/22462
Processados 18000/22462
Processados 18000/22462
Processados 19000/22462
Processados 19000/22462
Processados 20000/22462
Processados 20000/22462
Processados 21000

In [22]:
# Codificar labels, pad/truncate e salvar (modelo SMIC)
from sklearn.preprocessing import LabelEncoder
import json
from pathlib import Path

le = LabelEncoder()
if not isinstance(OUTPUT_DIR, Path):
    OUTPUT_DIR = Path(str(OUTPUT_DIR))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not frames_all or not labels_all:
    print('Nenhuma sequência/processamento disponível. Execute a célula de processamento antes de codificar e salvar.')
else:
    y_encoded = le.fit_transform(labels_all)
    print('Labels originais:', set(labels_all))
    print('Labels codificados:', set(y_encoded))

    padded = []
    for seq in frames_all:
        if seq.shape[0] < MAX_FRAMES:
            pad = np.zeros((MAX_FRAMES - seq.shape[0], seq.shape[1], seq.shape[2]), dtype=seq.dtype)
            new_seq = np.vstack([seq, pad])
        else:
            new_seq = seq[:MAX_FRAMES]
        padded.append(new_seq)

    X = np.array(padded)
    y = np.array(y_encoded)
    print('Shape X:', X.shape, 'Shape y:', y.shape)

    np.save(OUTPUT_DIR / 'X.npy', X)
    np.save(OUTPUT_DIR / 'y.npy', y)
    mapping = {str(cls): int(val) for cls, val in zip(le.classes_, le.transform(le.classes_))}
    with open(OUTPUT_DIR / 'label_mapping.json', 'w', encoding='utf-8') as f:
        json.dump(mapping, f, indent=2, ensure_ascii=False)
    np.save(OUTPUT_DIR / 'X_padded.npy', X)
    print('Salvo em', OUTPUT_DIR)


Labels originais: {'surprise', 'anger', 'sad', 'fear', 'happy', 'disgust'}
Labels codificados: {0, 1, 2, 3, 4, 5}
Shape X: (468, 16, 112, 112) Shape y: (468,)
Salvo em ../../data/SPOS/processed


In [ ]:
# Per-type padding (opcional) — só executa se `meta` existir
from sklearn.preprocessing import LabelEncoder
import json
from pathlib import Path
from collections import defaultdict as _dd

OUT = globals().get('OUTPUT_DIR', globals().get('OUTPUT_ROOT', Path('./')))

if 'meta' not in globals() or not frames_all:
    print("`meta` não encontrado ou `frames_all` vazio — pular per-type padding. Use a célula anterior que já salvou `X.npy`/`y.npy` ou reconstrua `meta` se precisar de padding por tipo.")
else:
    # Normalize frames_all entries: ensure shape (n_frames, H, W) and H,W == IMG_SIZE
    normalized = []
    converted_color = 0
    resized_count = 0
    expanded_count = 0
    empty_count = 0
    for seq in frames_all:
        arr = np.array(seq)
        # handle empty sequences defensively
        if arr.size == 0:
            empty_count += 1
            # create an empty array with zero frames but correct spatial size
            arr = np.zeros((0, IMG_SIZE[0], IMG_SIZE[1]), dtype=np.float32)
            normalized.append(arr)
            continue
        # If frames are color (n, H, W, C), convert to grayscale by averaging channels
        if arr.ndim == 4:
            converted_color += 1
            arr = np.mean(arr, axis=-1)
        # If a single image (H, W), expand dims to (1, H, W)
        if arr.ndim == 2:
            expanded_count += 1
            arr = np.expand_dims(arr, 0)
        # Ensure spatial size matches IMG_SIZE; resize frames if needed
        if (arr.shape[1], arr.shape[2]) != tuple(IMG_SIZE):
            frames_resized = []
            for f in arr:
                try:
                    f2 = cv2.resize(f, IMG_SIZE)
                except Exception:
                    f2 = cv2.resize(f.astype(np.uint8), IMG_SIZE)
                frames_resized.append(f2)
            arr = np.stack(frames_resized)
            resized_count += 1
        # cast to float32 and normalize to [0,1] if values appear to be in 0-255
        arr = arr.astype(np.float32)
        try:
            if arr.size and arr.max() > 1.0:
                arr = arr / 255.0
        except Exception:
            # if arr is empty or unexpected, skip normalization
            pass
        normalized.append(arr)
    # replace frames_all with normalized sequences for consistent padding
    frames_all = normalized

    GLOBAL_MAX = 69
    padded = []
    types = []
    for seq, m in zip(frames_all, meta):
        if isinstance(m, dict) and m.get('type'):
            t = m['type'].lower()
        else:
            t = 'unknown'
        if t.startswith('posed'):
            max_f = globals().get('MAX_FRAMES_POSED', 28)
        elif t.startswith('spont'):
            max_f = globals().get('MAX_FRAMES_SPONT', 69)
        else:
            max_f = GLOBAL_MAX
        # ensure seq is a numpy array with float32 dtype
        seq = np.array(seq, dtype=np.float32)
        # if sequence has 0 frames, create zero-padded sequence of length max_f
        if seq.shape[0] == 0:
            new_seq = np.zeros((max_f, IMG_SIZE[0], IMG_SIZE[1]), dtype=np.float32)
        else:
            if seq.shape[0] < max_f:
                pad = np.zeros((max_f - seq.shape[0], seq.shape[1], seq.shape[2]), dtype=np.float32)
                new_seq = np.vstack([seq, pad])
            else:
                new_seq = seq[:max_f]
        padded.append(new_seq)
        types.append(t)

    # Now group by type and save per-type arrays (so stacking succeeds even with different frame lengths)
    le = LabelEncoder()
    y_all = le.fit_transform([lbl for lbl in labels_all])

    by_type = _dd(list)
    y_by_type = _dd(list)
    for idx, (seq_arr, t) in enumerate(zip(padded, types)):
        t_safe = str(t).strip().lower().replace(' ', '_')[:50]
        by_type[t_safe].append(seq_arr)
        y_by_type[t_safe].append(int(y_all[idx]))

    summary = {}
    for t, seqs in by_type.items():
        Xt = np.array(seqs, dtype=np.float32)
        yt = np.array(y_by_type[t], dtype=np.int32)
        fname_x = OUT / f'X_per_type_{t}.npy'
        fname_y = OUT / f'y_per_type_{t}.npy'
        np.save(fname_x, Xt)
        np.save(fname_y, yt)
        summary[t] = {'n_sequences': int(Xt.shape[0]), 'n_frames': int(Xt.shape[1]), 'frame_shape': [int(s) for s in Xt.shape[2:]]}
        print(f"Salvo tipo='{t}' -> X: {fname_x} (shape={Xt.shape}), y: {fname_y} (shape={yt.shape})")

    # Save the label mapping and a small summary
    mapping = {str(cls): int(val) for cls, val in zip(le.classes_, le.transform(le.classes_))}
    with open(OUT / 'label_mapping_per_type.json', 'w', encoding='utf-8') as f:
        json.dump(mapping, f, indent=2, ensure_ascii=False)
    with open(OUT / 'per_type_summary.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print('Conversões: color->gray:', converted_color, 'resized sequences:', resized_count, 'expanded single-image seqs:', expanded_count, 'empty seqs:', empty_count)
    print('Salvo em', OUT)


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (468,) + inhomogeneous part.